In [ ]:


export=False


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import time
import functools as ft
from IPython.display import display
pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_csm = path_prod / 'Chamber Study Missions' / date.today().strftime('%Y') / '2026 peer region tour'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())



In [ ]:


# Assign geographies
sacog_state = ["Sacramento, CA", "Yuba City, CA"]

other_peers = ["Atlanta, GA", "Cleveland, OH", "Minneapolis, MN", "Kansas City, KS", "Portland, OR", "Seattle, WA"]

# More classifiers
sacog = ["Yuba City", "Sacramento"]

sample_type = 'Zillow'
year_start = 2000
year_end = 2025
geography = 'MSA'



In [ ]:


indicator = 'Cost_1'

df_about = write_about(sample_type=sample_type
                       , indicator=indicator
                       , year_start=year_start
                       , year_end=year_end
                       , path_config0=path_config0
                       , geography=geography)
print("About page documentation table:")
display(df_about)
print()

# Load CSV file
file_in = path_raw / 'Zillow' / "Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
df_cost1 = pd.read_csv(file_in)
df_cost1 = df_cost1[df_cost1['RegionType'] == 'msa']
df_cost1 = pd.melt(df_cost1, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], var_name='date_', value_name='Price')

# Subset and rename cols
df_cost1 = df_cost1[['RegionName', 'StateName', 'date_', 'Price']]
df_cost1.columns = ['MSA', 'State', 'date_', 'Price']
df_cost1 = df_cost1[df_cost1['MSA'].isin(sacog_state + other_peers)]



# Calculate median prices for each MSA

file_weights = path_main / 'Vibrant and Inclusive Places' / 'People and Community' / 'Pop and Demographics' / 'DOF_E5_and_E8_Jurisdictions.xlsx'
df_weights = pd.read_excel(file_weights)

df_weights = df_weights[df_weights['MPO'] == 'SACOG']
df_weights = df_weights.rename(columns={'Housing Units':'Households'})

conditions = [
    df_weights['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Yolo'])
    , df_weights['County'].isin(['Sutter', 'Yuba'])
]

choices = ['Sacramento, CA', 'Yuba City, CA']

df_weights['MSA'] = np.select(conditions, choices, 'no')

df_weights = df_weights[['MSA', 'Year', 'Households']]
df_weights = df_weights.groupby(['MSA', 'Year'], as_index=False)['Households'].sum()

df_cost1['date_'] = pd.to_datetime(df_cost1['date_'])
df_cost1['Year'] = df_cost1['date_'].dt.year

df_cost1 = df_cost1.merge(df_weights, on=['MSA', 'Year'], how='left')
df_cost1.loc[df_cost1['MSA'] == 'Yuba City, CA' , 'MSA'] = 'SACOG'
df_cost1.loc[df_cost1['MSA'] == 'Sacramento, CA', 'MSA'] = 'SACOG'
df_cost1['Households'] = df_cost1['Households'].fillna(1)
df_cost1 = df_cost1.reset_index(drop=True)
wm = lambda x: np.average(x, weights = df_cost1.loc[x.index, "Households"]) # weighted average (or Population or Households)
df_cost1 = df_cost1.groupby(['MSA', 'date_'], as_index=False, sort=False).agg(Price=('Price', wm))



# Concatenate data
df_cost1 = df_cost1.sort_values(by=['MSA', 'date_'], ascending = [True, False])
df_cost1 = df_cost1[['MSA', 'date_', 'Price']]
df_cost1 = df_cost1.reset_index(drop=True)

df_cost1_yr = df_cost1.copy()

df_cost1_yr['Year'] = df_cost1_yr['date_'].dt.year
df_cost1_yr = df_cost1_yr.drop('date_', axis = 1)
df_cost1_yr = df_cost1_yr.groupby(['MSA', 'Year'], as_index = False)['Price'].mean()
df_cost1_yr = df_cost1_yr.sort_values(by=['MSA', 'Year'], ascending = [True, False])

display(df_cost1_yr.head(6))


df_cost1['date_'] = df_cost1['date_'].astype(str)



# Export

if export:
    
    file_out = path_csm / f"{indicator} Region Zillow_ChamberStudy2026.xlsx"
    with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
        df_about   .to_excel(writer, index = False, sheet_name = 'About'  , header=False)
        df_cost1   .to_excel(writer, index = False, sheet_name = 'Monthly'              )
        df_cost1_yr.to_excel(writer, index = False, sheet_name = 'Annual'               )
    print('exported successfully')



df_plot = df_cost1_yr.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

x = 'year'
y = 'price'
color = 'msa'
labels = 'msa'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price by MPO')

    
fig.show()

In [ ]:
indicator = 'Cost_2'

df_about = write_about(sample_type=sample_type
                       , indicator=indicator
                       , year_start=year_start
                       , year_end=year_end
                       , path_config0=path_config0
                       , geography=geography)
print("About page documentation table:")
display(df_about)
print()

# Load CSV file
file_in = path_raw / 'Zillow' / "Metro_zori_uc_sfrcondomfr_sm_month.csv"
df_cost2 = pd.read_csv(file_in)
df_cost2 = df_cost2[df_cost2['RegionType'] == 'msa']
df_cost2 = pd.melt(df_cost2, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], var_name='date_', value_name='Price')

# Subset and rename cols
df_cost2 = df_cost2[['RegionName', 'StateName', 'date_', 'Price']]
df_cost2.columns = ['MSA', 'State', 'date_', 'Price']
df_cost2 = df_cost2[df_cost2['MSA'].isin(sacog_state + other_peers)]



# Calculate median prices for each MSA

file_weights = path_main / 'Vibrant and Inclusive Places' / 'People and Community' / 'Pop and Demographics' / 'DOF_E5_and_E8_Jurisdictions.xlsx'
df_weights = pd.read_excel(file_weights)

df_weights = df_weights[df_weights['MPO'] == 'SACOG']
df_weights = df_weights.rename(columns={'Housing Units':'Households'})

conditions = [
    df_weights['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Yolo'])
    , df_weights['County'].isin(['Sutter', 'Yuba'])
]

choices = ['Sacramento, CA', 'Yuba City, CA']

df_weights['MSA'] = np.select(conditions, choices, 'no')

df_weights = df_weights[['MSA', 'Year', 'Households']]
df_weights = df_weights.groupby(['MSA', 'Year'], as_index=False)['Households'].sum()

df_cost2['date_'] = pd.to_datetime(df_cost2['date_'])
df_cost2['Year'] = df_cost2['date_'].dt.year

df_cost2 = df_cost2.merge(df_weights, on=['MSA', 'Year'], how='left')
df_cost2.loc[df_cost2['MSA'] == 'Yuba City, CA' , 'MSA'] = 'SACOG'
df_cost2.loc[df_cost2['MSA'] == 'Sacramento, CA', 'MSA'] = 'SACOG'
df_cost2['Households'] = df_cost2['Households'].fillna(1)
df_cost2 = df_cost2.reset_index(drop=True)
wm = lambda x: np.average(x, weights = df_cost2.loc[x.index, "Households"]) # weighted average (or Population or Households)
df_cost2 = df_cost2.groupby(['MSA', 'date_'], as_index=False, sort=False).agg(Price=('Price', wm))



# Concatenate data
df_cost2 = df_cost2.sort_values(by=['MSA', 'date_'], ascending = [True, False])
df_cost2 = df_cost2[['MSA', 'date_', 'Price']]
df_cost2 = df_cost2.reset_index(drop=True)

df_cost2_yr = df_cost2.copy()

df_cost2_yr['Year'] = df_cost2_yr['date_'].dt.year
df_cost2_yr = df_cost2_yr.drop('date_', axis = 1)
df_cost2_yr = df_cost2_yr.groupby(['MSA', 'Year'], as_index = False)['Price'].mean()
df_cost2_yr = df_cost2_yr.sort_values(by=['MSA', 'Year'], ascending = [True, False])

display(df_cost2_yr.head(6))


df_cost2['date_'] = df_cost2['date_'].astype(str)


# Export

if export:
    
    file_out = path_csm / f"{indicator} Region Zillow_ChamberStudy2026.xlsx"
    with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
        df_about   .to_excel(writer, index = False, sheet_name = 'About'  , header=False)
        df_cost2   .to_excel(writer, index = False, sheet_name = 'Monthly'              )
        df_cost2_yr.to_excel(writer, index = False, sheet_name = 'Annual'               )
    print('exported successfully')


df_plot = df_cost2_yr.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

x = 'year'
y = 'price'
color = 'msa'
labels = 'msa'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price by MPO')

    
fig.show()